In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
df = pd.read_csv("merged_data.csv")
df_clean = df.dropna()
df_clean = df_clean[(df_clean["Diabetes_Status"] != 3) & (df_clean["Diabetes_Status"] != 9)]
df_clean = df_clean.drop(columns=['SEQN'])
print(df_clean.count())

Age                  3167
Gender               3167
Height_cm            3167
Weight_kg            3167
Waist_cm             3167
Hip_cm               3167
Systolic_BP          3167
Diastolic_BP         3167
HbA1c                3167
FBS                  3167
Cholesterol_Total    3167
HDL                  3167
Diabetes_Status      3167
dtype: int64


In [8]:
train, valid, test = np.split(df_clean.sample(frac=1), [int(0.6*len(df_clean)), int(0.8*len(df_clean))])

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


# **Raw Dataset Training**
all values are raw that contains NaN.

GENDER:
*   MALE: 1,
*   FEMALE: 2

DIABETES STATS:
*   YES: 1
*   NO: 2





In [9]:
print((df_clean['Diabetes_Status'].value_counts()))
print(df_clean.columns)

Diabetes_Status
2.0    2785
1.0     382
Name: count, dtype: int64
Index(['Age', 'Gender', 'Height_cm', 'Weight_kg', 'Waist_cm', 'Hip_cm',
       'Systolic_BP', 'Diastolic_BP', 'HbA1c', 'FBS', 'Cholesterol_Total',
       'HDL', 'Diabetes_Status'],
      dtype='object')


In [10]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import StandardScaler

def scale_dataset(dataframe, oversample=False):
	X = dataframe[dataframe.columns[:-1]].values
	y = dataframe[dataframe.columns[-1]].values

	scaler = StandardScaler()
	X = scaler.fit_transform(X)

	if oversample:
		ros = RandomOverSampler()
		X, y = ros.fit_resample(X, y)

	# puttig them side by side / hstack (horizontal stack) like combine
	# np is sensitive to 2D array, convert the y into 2D array
	data = np.hstack((X, np.reshape(y, (-1, 1))))

	return data, X, y

In [11]:
train, X_train, y_train = scale_dataset(train,oversample=True)
valid, X_valid, y_valid = scale_dataset(valid,oversample=False)
test, X_test, y_test = scale_dataset(test,oversample=False)

In [12]:
print(len(y_train[y_train==1]))
print(len(y_train[y_train==2]))


1663
1663


K-NEAREST NEIGHBORS

In [13]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=1)
knn_model.fit(X_train, y_train)

# PREDICTIONS
y_pred = knn_model.predict(X_test)


In [14]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         1.0       0.62      0.49      0.55        78
         2.0       0.93      0.96      0.94       556

    accuracy                           0.90       634
   macro avg       0.78      0.72      0.75       634
weighted avg       0.89      0.90      0.90       634



NAIVE BAYES

In [15]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model = nb_model.fit(X_train, y_train)

# PREDICTIONS
y_pred = nb_model.predict(X_test)

# CLASSIFICATION REPORT
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         1.0       0.66      0.78      0.72        78
         2.0       0.97      0.94      0.96       556

    accuracy                           0.92       634
   macro avg       0.82      0.86      0.84       634
weighted avg       0.93      0.92      0.93       634



LOGISTIC REGRESSION

In [16]:
from sklearn.linear_model import LogisticRegression

lg_model = LogisticRegression()
lg_model = lg_model.fit(X_train, y_train)

# PREDICTIONS
y_pred = lg_model.predict(X_test)

# CLASSIFICATION REPORT
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         1.0       0.58      0.83      0.68        78
         2.0       0.98      0.91      0.94       556

    accuracy                           0.90       634
   macro avg       0.78      0.87      0.81       634
weighted avg       0.93      0.90      0.91       634



SUPPORT VECTOR MACHINES

In [17]:
from sklearn.svm import SVC

# There are many params for this, tweak and try them
svm_model = SVC()
svm_model = svm_model.fit(X_train, y_train)

# PREDICTIONS
y_pred = svm_model.predict(X_test)

# CLASSIFICATION REPORT
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         1.0       0.59      0.81      0.68        78
         2.0       0.97      0.92      0.95       556

    accuracy                           0.91       634
   macro avg       0.78      0.87      0.82       634
weighted avg       0.93      0.91      0.91       634



RANDOM FORESTS

In [18]:
from sklearn.ensemble import RandomForestClassifier

forest_model = RandomForestClassifier(random_state=1)
forest_model.fit(X_train, y_train)

# PREDICTIONS
y_pred = forest_model.predict(X_test)

# CLASSIFICATION REPORT
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         1.0       0.77      0.71      0.74        78
         2.0       0.96      0.97      0.97       556

    accuracy                           0.94       634
   macro avg       0.87      0.84      0.85       634
weighted avg       0.94      0.94      0.94       634



[[ 0.2916712   0.74413698  0.90366969 ...  0.47004449  0.25316792
   0.75485497]
 [ 0.50200346 -1.38614363 -1.10659903 ... -0.3849608  -0.82422728
  -0.46381292]
 [-1.06017858  0.54597134 -1.10659903 ... -0.15177754  0.20632465
   0.61148228]
 ...
 [-0.30183778  1.28909248 -1.10659903 ... -0.07404978 -0.6368542
  -1.53910813]
 [-0.79118222  1.53679953 -1.10659903 ...  0.36640749 -1.03502199
   0.39642324]
 [ 1.56368058  0.69459557 -1.10659903 ... -0.09995904 -0.00447006
  -1.32404909]]


In [20]:
# new patient example (must follow same feature order used in training!)

new_patient = [[43, 2, 179.5, 86.9,98.3,102.9,135,98,8.7,113,264,45]]

print("Prediction:", forest_model.predict(new_patient))
print("Probabilities:", forest_model.predict_proba(new_patient))


Prediction: [2.]
Probabilities: [[0.35 0.65]]
